# Post processing and visualizing mandyoc data

This notebook reads the NetCDF files created by tapioca (`meshes2netcdf.jl`): `temperature.nc`, `density.nc`, `strain.nc`, `lithology.nc`, ... in the scenario folder, and plots them with `plot_property` of `functions/mandyocIO.py`.

The following notebook allows the user to visualize a given mandyoc output (e.g.: `'temperature'`, `'strain_rate'`, `'lithology'`, etc) according to a given number of time instants (e.g.: `instants = [0, 10, 20, 30]` Myr).

For a given instant in the `instants` the script will find the closest time step to this instant and plot the respective output.

The crustal thickness is measured counting the crust cells of `lithology.nc` (`measure_crustal_thickness` and `measure_margins_width` of `functions/mandyocIO.py`).


In [ ]:
#increase the display size to 95% of window width
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
display(HTML("<style>.output_result { max-width:95% !important; }</style>"))


import os
import sys
import string
import numpy as np
import matplotlib.pyplot as plt

path = os.getcwd().split('/')
machine_path = '/'+path[1]+'/'+path[2] #cat the /home/user/ or /Users/user from system using path

sys.path.append(f"{machine_path}/opt/post_processing") #functions/mandyocIO.py, if the notebook is not run from post_processing
from functions.mandyocIO import (read_tapioca_datasets, read_params, read_times_steps, find_nearest, trim_axs,
                                 measure_crustal_thickness, measure_margins_width, plot_property)

##File of fancy plot style. To use des.mplstyle file you must have an LaTeX distribution in your system
#plt.style.use(machine_path+'/Mestrado/cenarios/mandyoc/scripts_gera_inputs_imgs/new_outputs/des.mplstyle')


# How to plot

## In the following cell, you must inform:
 - The path to your `scenario` (folder with the `.nc` files created by tapioca) without `/OS_home/username` because this is collected by `machine_path`.
     - Ex.: `/mandyoc/experiments/my_experiment/`
 - The mandyoc property `prop` that you want to visualize uncommenting one the commented options.
     - `prop = 'lithology'` is the default.
 - The Mandyoc layer codes of the crust (`crust_codes`) and the colors of the layer codes (`lithology_colors`) of your model.
 - The values in list `instants` containing the instants that you want to visualize.
     - Ex: `instants = [0, 10, 20, 30]` #Myr

In [ ]:
#Set scenario path

#scenario = '/Documents/aguiasala/M45/M45/'
#scenario = '/Documents/Simulations/M60_1x1/'
#scenario = '/Documents/Simulations/Invertedmargin/'
# scenario = '/Documents/Simulations/MR_dk/'
# scenario = '/Desktop/teste/'
# scenario = '/Desktop/teste3/'
scenario = '/Desktop/teste4/'
# scenario = '/opt/teste/'
##############################################################################################################
#Creating path to the experiment

fpath = machine_path + scenario
print(fpath)

##############################################################################################################
#Set mandyoc property

# prop = 'density'
# prop = 'radiogenic_heat'
prop = 'lithology'
#prop = 'pressure'
#prop = 'strain'
# prop = 'strain_rate'
#prop = 'temperature'
# prop = 'temperature_anomaly'
# prop = 'surface'
# prop = 'viscosity'

##############################################################################################################
#Mandyoc layer codes (order of the layers in interfaces.txt, 0 = bottom layer) of the crust,
#used to measure the crustal thickness counting the crust cells of lithology.nc

crust_codes = [4, 5, 6, 7, 8, 9] #double keel: lower crust 1, seed, lower crust 2, upper crust, decolement, sediments
# crust_codes = [4, 5] #accordion (AR) and rifting: lower crust, upper crust
# crust_codes = [4, 5, 6] #accordion (AR) with the basaltic (oceanic) crust

##############################################################################################################
#Colors of the Mandyoc layer codes in the lithology plot: a key of LITHOLOGY_COLORS (functions/mandyocIO.py)
#or a list with one color per code (order of the layers in interfaces.txt, 0 = bottom layer, the air is the last one)

lithology_colors = 'double_keel'
# lithology_colors = 'rifting'
# lithology_colors = 'accordion'

##############################################################################################################
#Set instants

#instants = [25,75,105]
#instants = [0,25, 35, 49,120]
# instants = [0,5,10,15]
# instants = [30, 100]
instants = [25, 75, 175, 230]
# instants = [0, 30, 60, 90, 120] #Myr
# instants = [0, 30, 60, 90, 120, 150]

##############################################################################################################
#Reading scenario basic informations

times, steps = read_times_steps(fpath)
Nx, Nz, Lx, Lz = read_params(fpath)
x = np.linspace(0, Lx/1000, Nx)
z = np.linspace(-Lz/1000.0, 0, Nz)

#Reading the .nc files used by plot_property (density: topography; temperature: isotherms; strain: lithology shading)
variables = ['density', 'temperature', 'strain']
if(prop not in ['lithology', 'temperature_anomaly', 'surface']):
    variables.append(prop)
dataset = read_tapioca_datasets(fpath, variables)

##############################################################################################################
#Set domain limits of the figure
#Aqui mudamos os limites de x e y de plotagem da figura!
if(prop != 'surface'):
    xlims = [0, Lx / 1.0e3]
    ylims = [-Lz / 1.0e3 + 40, 40]
    # ylims = [-80, 40]
else:
    xmin = 0 #+ 200
    xmax = Lx / 1.0E3 #- 200
    xlims = [xmin, xmax]
    ylims = [-6, 6]

##############################################################################################################
#Creating figure and subplots
plt.close('all')

if(len(instants) == 1):
    fig, axs = plt.subplots(len(instants), 1, constrained_layout=True, figsize=(12,6), sharex=True, sharey=True)
    axs = [axs]
else:
    fig, axs = plt.subplots(len(instants), 1, constrained_layout=True, figsize=(12,12), sharex=True, sharey=True)
    axs = trim_axs(axs, len(instants))


alphabet = list(string.ascii_lowercase)
letters = alphabet[0: len(instants)]

##############################################################################################################
#Plot data
plot_thickness = True
# plot_thickness = False

for instant, ax in zip(instants, axs):
    idx = find_nearest(times, instant)
    step = steps[idx]
    plot_property(dataset.isel(time=idx), prop, xlims, ylims, fpath, fig, ax,
                  print_time=False,
                  isotherms=[550, 800, 1300], label_isotherms=(ax == axs[0]),
                  plot_colorbar=(ax == axs[-1]),
                  bbox_to_anchor=(0.80, 0.09, 0.18, 0.16), #legend of the lithology: lower right corner at (0.98, 0.09)
                  lithology_colors=lithology_colors)

    if(plot_thickness == True):
        if(instant == 25): #instant to measure margin width
            x_begin = 200.0
            x_end = 700.0
            calc_margin_width = True
            
        elif(instant == 230): #instant to measure margin width
            x_begin = 200.0
            x_end = 700.0
            # x_begin = 900.0
            # x_end = 1400.0
            calc_margin_width = True

        else: #orogeny instants
            x_begin = 200.0
            x_end = 800.0
            calc_margin_width = False
            
        x_pos, topo_max_thickness, local_crustal_thickness, lc_base, topography, total_crustal_thickness = measure_crustal_thickness(fpath, step, Nx, Nz, Lx, Lz, x_begin, x_end, crust_codes=crust_codes)
        
        # ax.plot(np.linspace(0, Lx/1000, Nx), lc_base, color='black', linestyle='--', linewidth=2.0, zorder=52)
        # ax.plot(np.linspace(0, Lx/1000, Nx), topo, color='black', linestyle='-', linewidth=2.0, zorder=52)
        ax.plot([x_pos, x_pos], [topo_max_thickness, -local_crustal_thickness], color='black', linestyle='--', linewidth=1.0, zorder=52)
        ax.text(x_pos+100, -local_crustal_thickness/2, f'{str(round(local_crustal_thickness, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)

        if(instant == 230 or instant == 25): #checking for regions with crustal thickness between 7 and 10 km
            h_crust = total_crustal_thickness #crustal thickness
            cond_thickness = (h_crust <= 10) & (h_crust >= 7)
            ax.plot(x[cond_thickness], lc_base[cond_thickness], 'o', color='xkcd:orange', zorder=52)

        if(calc_margin_width == True): #Measuring margin width
            if(instant == 25):
                xl_begin = 780
                xl_end = 900

                xr_begin = 950
                xr_end = 1100
                
            elif(instant == 230):
                xl_begin = 300
                xl_end = 750

                xr_begin = 800
                xr_end = 1350

            marginl_begin, marginl_end, marginl_len, marginr_begin, marginr_end, marginr_len = measure_margins_width(fpath, step, Nx, Nz, Lx, Lz, xl_begin, xl_end, xr_begin, xr_end, crust_codes=crust_codes)

            ax.plot([marginl_begin, marginl_end], [10,10], color='orange', linestyle='-', linewidth=4.0, zorder=52)
            ax.text(marginl_begin + marginl_len/2, 12, f'{str(round(marginl_len, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)

            ax.plot([marginr_begin, marginr_end], [10, 10], color='blue', linestyle='-', linewidth=4.0, zorder=52)
            ax.text(marginr_begin - marginr_len/2, 12, f'{str(round(marginr_len, 2))} km', fontsize=12, ha='center', va='bottom', zorder=52)

    #print(steps[idx], times[idx], instant)
    
##############################################################################################################


#Setting plot details

fsize = 14
axs[-1].set_xlabel('x [km]', fontsize=fsize)
for ax, letter, instant in zip(axs, letters, instants):
    #letter of the subplot and instant outside the plot, in the upper left corner
    label_text = letter + ') ' + str(instant) + ' Myr'
    ax.text(0.0,
            1.02,
            label_text,
            color='k',
            fontsize=fsize,
            horizontalalignment='left',
            verticalalignment='bottom',
            transform=ax.transAxes)
    
    if(prop != 'surface'):
        ax.set_ylabel('Depth [km]', fontsize=fsize)
        ax.tick_params(axis='both', labelsize=fsize)
        axs[-1].tick_params(axis='both', labelsize=fsize)
        ax.set_yticks(np.linspace(ylims[0], 0, 5))
    else:
        ax.set_ylabel('Topogarphy [km]', fontsize=fsize)
        ax.tick_params(axis='both', labelsize=fsize)
        axs[-1].tick_params(axis='both', labelsize=fsize)
        ax.set_yticks(np.linspace(ylims[0], ylims[1], 7))
        ax.grid('-k', alpha = 0.7)

axs[0].set_xlim(xlims)
axs[0].set_ylim(ylims)
# axs[0].set_ylim([-40,40])    
##############################################################################################################
#Saving figure

figname = scenario.split('/')[-2] + '_'+prop
print(figname)

#This will save inside the notebook directory
#fig.savefig(figname+'.pdf', dpi=400)
#fig.savefig(figname+'.png', dpi=400)

#This will save inside the experiment directory
#fig.savefig(fpath+figname+'.pdf', dpi=400)
fig.savefig(fpath+figname+'.png', dpi=400)
#fig.savefig(fpath+figname+'.jpeg', dpi=400)